# 02 - Return Prediction

Build and check the cohort defined in notebook 01, compare activity rules with two models, and evaluate the selected ranking on a later holdout.

In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()
if not (project_dir / "src" / "data_pipeline.py").exists():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

## Build and check the cohort

In [2]:
from src.data_pipeline import run_pipeline

run_pipeline()

Saved 991,833 eligible users to data/processed/retention_cohort.parquet
Returned during days 15-44: 34,340 (3.5%)


In [3]:
import pandas as pd

cohort_path = project_dir / "data" / "processed" / "retention_cohort.parquet"
cohort = pd.read_parquet(cohort_path)
cohort.head()

,user_id,first_event_time,events_first_14_days,active_days_first_14_days,high_intent_actions_first_14_days,returned_days_15_44
0,1185234,2015-05-05 20:31:20.888000+00:00,145,10,13,1
1,350566,2015-05-06 15:06:38.741000+00:00,871,11,134,1
2,111566,2015-05-05 16:31:50.214000+00:00,1,1,0,1
3,778052,2015-05-14 19:02:41.520000+00:00,1,1,0,1
4,1210904,2015-05-04 17:05:43.320000+00:00,2,1,0,1


In [4]:
from src.train_model import FEATURES, TARGET
assert cohort["user_id"].is_unique
assert cohort[TARGET].notna().all()
assert set(cohort[TARGET].unique()).issubset({0, 1})

print(f"Cohort: {len(cohort):,} visitors")
print(cohort[TARGET].value_counts().rename(index={0: "no return", 1: "returned"}))
print(f"Return rate: {cohort[TARGET].mean():.1%}")
cohort.groupby(TARGET)[FEATURES].mean().T.rename(
    columns={0: "No return", 1: "Returned"}
)

Cohort: 991,833 visitors
returned_days_15_44
no return    957493
returned      34340
Name: count, dtype: int64
Return rate: 3.5%


returned_days_15_44,No return,Returned
events_first_14_days,1.724278,4.356931
active_days_first_14_days,1.084147,1.578567
high_intent_actions_first_14_days,0.049408,0.248253


## Feature distributions

Returners have higher average early activity in this cohort. The counts are skewed, so these means alone do not show how well a ranking will perform.

In [5]:
cohort[FEATURES].describe()

,events_first_14_days,active_days_first_14_days,high_intent_actions_first_14_days
count,991833.000000,991833.000000,991833.000000
mean,1.815428,1.101265,0.056293
std,6.053983,0.461809,1.017989
min,1.000000,1.000000,0.000000
25%,1.000000,1.000000,0.000000
50%,1.000000,1.000000,0.000000
75%,2.000000,1.000000,0.000000
max,1421.000000,15.000000,355.000000


## Evaluation setup

Select the ranking with the highest validation average precision (AP), then evaluate that choice on a later holdout.

Training labels must be observable by July 3, 2015. Validation scoring runs July 3-7; its outcomes are complete by August 8, when holdout scoring starts. Waiting for complete outcomes before each stage leaves short evaluation periods within the available log. All cutoffs use UTC.

## Comparisons

Event count and active days test whether a simple rule is enough. Logistic regression and XGBoost use the three activity features checked above.

AP measures ranking across thresholds. The return rate provides a random-ranking reference. The top 10% is an illustrative capacity limit. For tied scores, metrics average over random selection at the cutoff.

In [6]:
from src.train_model import train_model
results = train_model()


Training users for comparison: 169,613
Validation users: 46,854
Validation share of eligible cohort: 4.7%
Validation scoring period (UTC): 2015-07-03 00:00:00+00:00 to 2015-07-08 00:00:00+00:00
Training labels observable by (UTC): 2015-07-03 00:00:00+00:00
Validation outcomes fully observable by (UTC): 2015-08-08 00:00:00+00:00
Validation return rate: 0.036
event count validation AP: 0.06874
active days validation AP: 0.07808
logistic regression validation AP: 0.09310
XGBoost validation AP: 0.09255
Selected ranking: logistic regression
Final training users: 538,406
Final holdout feature-ready period (UTC): 2015-08-08 00:00:06.723000+00:00 to 2015-08-18 02:59:46.326000+00:00
Final training labels observable by (UTC): 2015-08-08 00:00:00+00:00
Final holdout users: 114,973
Final holdout return rate (random-ranking AP baseline): 0.027
logistic regression holdout: AP 0.079, top-10% precision 7.8%, recall 28.8%
event count holdout: AP 0.054, top-10% precision 6.3%, recall 23.3%
active days h

## Interpretation

Validation selected logistic regression; its AP and XGBoost's both round to 0.093. On the later holdout, logistic regression achieved AP of 0.079 versus 0.072 for active days and 0.054 for event count.

At the illustrative top-10% cutoff, logistic regression and active-day ranking both had 7.8% precision, capturing 28.8% and 28.7% of returners respectively. Active days nearly matched logistic regression at the 10% cutoff.

This comparison uses three activity features and short evaluation periods. Performance outside this period is untested. Activity-rule benchmarks were added after inspecting earlier holdout results. The outcome is observed return; campaign impact requires a randomized experiment.